# 01 — Stylized facts of equity returns

**Companion:** [docs/concept-guide.md](../docs/concept-guide.md) §§1–3 · **Timeboxes:** S1–S3 in [ROADMAP.md](../ROADMAP.md)

**Goal:** verify four empirical regularities on ~20 years of S&P 500 daily returns, then build the rolling-volatility estimator that GARCH will replace tomorrow.

The rules:

1. **You write every analysis cell.** Cells marked `TODO(you)` are yours; each states its DONE criterion.
2. **Checkpoints are your oracle.** After each task, compare your numbers against the expected magnitudes. Outside the range → debug before moving on.
3. **Hints are folded and tiered** (concept → API → pseudocode). Open them deliberately, one rung at a time — the struggle is the learning.
4. **After each checkpoint, write the one interpretation sentence it asks for.** On Sunday your README assembles itself from these.

In [ ]:
# GIVEN — the only free code in this notebook.
# autoreload: your edits to src/garch_lab/*.py take effect here without restarting the kernel.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from garch_lab.data import load_prices

px = load_prices("^GSPC", start="2005-01-01")  # downloads once, then reads data/*.csv
px.tail()

## §1 From prices to returns — guide §1

Prices are non-stationary: statistics computed on them do not converge to anything interpretable. Returns are approximately stationary and scale-free — *they* are the object of study.

**Think before coding:**

- Q1. What would the sample mean of the S&P's price *level* over 2005–2026 actually estimate? Why is stationarity the license to do statistics at all? (You know this from statistical mechanics: which property lets a single realized history stand in for the ensemble?)
- Q2. Log vs simple returns: what exactly do you gain on the time axis, and what do you give up across assets? (Guide §1.2.)

**Task** — implement `log_returns` in [src/garch_lab/returns.py](../src/garch_lab/returns.py).

**DONE when:** `pytest tests/test_returns.py` is green. Then compute `r` and make two stacked plots: price level on top, daily log returns below.

<details><summary>Hint 1 — concept</summary>

A log return is a first difference of log prices. Decide which date should label each return — the test has an opinion, and so does every merge you will ever do against these dates.

</details>

<details><summary>Hint 2 — API</summary>

`np.log`, `Series.diff()`, `Series.dropna()`.

</details>

In [ ]:
# TODO(you): make tests/test_returns.py green first, then:
#   r = log_returns(px)
#   two stacked plots: price level (top), daily log returns (bottom)
# DONE when: the tests pass and you can SEE 2008 and 2020 in the returns panel.

### ✓ Checkpoint 1

- Observations: n ≈ 5,400 ± 100 (grows with your download date).
- Daily σ̂ ≈ 0.8–1.4% · daily mean ≈ 0.02–0.05% — note the order-of-magnitude gap; §3 will use it.
- The return plot already shows 2008 and 2020 as *clusters* of large bars, of both signs.

➜ **One sentence:** what do you see in the return plot that an iid model forbids?

_Your sentence:_

## §2.1 Fat tails — guide §2.1

**Predict before you compute:**

- Q1. With your daily σ̂, how many |r| > 4σ days should a *Gaussian* produce in ~5,300 observations? (P(|Z| > 4) ≈ 6.3×10⁻⁵.) Write your number down, then count the real ones.
- Q2. Kurtosis is a fourth-moment estimator. What dominates its own sampling variance — and what does that predict about its stability across subsamples?

**Task:** (a) excess kurtosis on the full sample and on each half separately; (b) count days with |r| > 4σ̂; (c) QQ plot of *standardized* returns vs the normal → export `figures/fig2_qq_tails.png`; (d) fit a Student-t by maximum likelihood, report ν̂; (e) Jarque–Bera.

**DONE when:** the checkpoint magnitudes hit and the figure is exported.

<details><summary>Hint 1 — concept</summary>

Standardize returns (mean 0, variance 1) before the QQ plot so the 45° line is the Gaussian reference.

</details>

<details><summary>Hint 2 — API traps</summary>

`scipy.stats.kurtosis` already returns EXCESS kurtosis — do not subtract 3 twice. `scipy.stats.t.fit` returns `(df, loc, scale)`, and `scale` is not σ (Var = scale²·ν/(ν−2)). `statsmodels.graphics.gofplots.qqplot(..., line="45")`. `scipy.stats.jarque_bera`.

</details>

In [ ]:
# TODO(you): kurtosis (full + two halves), 4-sigma day count,
#            QQ plot -> figures/fig2_qq_tails.png, t-fit nu-hat, Jarque-Bera

### ✓ Checkpoint 2

- κ_excess ∈ [5, 30], and visibly different between the two halves — **that instability is a finding about the tails, not a bug in your code.**
- 4σ days: the Gaussian expects < 1 in this sample; you should count ≥ 10.
- ν̂ ∈ [3, 6] — heavy enough that the fourth moment is borderline undefined, consistent with the instability you just saw.
- QQ: the S-curve — below the line lower-left, above it upper-right.
- Jarque–Bera p ≈ 0.

➜ **One sentence:** why is that p-value nearly uninformative at n ≈ 5,000, and what carries the information instead? (Guide §2.1.)

_Your sentence:_

## §2.2 No linear autocorrelation in returns — guide §2.2

- Q. Suppose ρ(1) were reliably +0.10. What trade exists, who runs it, and what does their activity do to ρ(1)?

**Task:** ACF of r out to lag 50, with the ±1.96/√n band.

**DONE when:** the plot exists and you can defend what the band assumes.

<details><summary>Hint — API</summary>

`statsmodels.graphics.tsaplots.plot_acf(..., lags=50, zero=False)` — the lag-0 bar equals 1.0 by definition and flattens everything else if you keep it.

</details>

In [ ]:
# TODO(you): ACF of r to lag 50, iid confidence band

### ✓ Checkpoint 3

- Essentially all lags inside the band; scattered marginal violations at roughly the 5% rate chance predicts.
- Worth writing down: the band assumes iid — heteroskedasticity (next section!) inflates the true sampling variance of ρ̂, so do not interpret stragglers.

➜ **One sentence:** "uncorrelated is not independent" — where could structure still hide?

_Your sentence:_

## §2.3 Volatility clustering — guide §2.3

- Q. The series r and r² carry exactly the same digits. How can one be white noise while the other has 100-day memory? In which *moment* does the dependence live — and what does that say about the decomposition r_t = σ_t · z_t?

**Task:** ACF of r² and of |r| out to lag 100. Then a two-panel figure — daily returns on top, ACF of |r| below → export `figures/fig1_clustering.png`.

**DONE when:** exported and the checkpoint hits.

In [ ]:
# TODO(you): ACF of r^2 and |r| to lag 100;
#            two-panel figure -> figures/fig1_clustering.png

### ✓ Checkpoint 4

- ρ̂(1) of |r| ≈ 0.2–0.35; positive and significant out to 50–100+ lags; decaying slowly.
- **This plot is the empirical license for the entire project** — the second moment is forecastable. Your README needs this figure.

➜ **One sentence:** Mandelbrot said it in 1963 — say it in your own words.

_Your sentence:_

## §2.4 The leverage effect — guide §2.4 · *(cuttable — ROADMAP cut #2)*

- Q. Tomorrow you will feed shocks into GARCH as ε². What information does squaring destroy — and what correlation would demonstrate that the destroyed information actually matters?

**Task:** Corr(r_t, |r_{t+k}|) for k = 1…20, as a bar plot.

<details><summary>Hint — API</summary>

`r.corr(r.abs().shift(-k))` inside a loop or comprehension — mind the direction of the shift: today's return against *future* magnitude.

</details>

In [ ]:
# TODO(you): leverage correlations, k = 1..20, bar plot

### ✓ Checkpoint 5

- Negative at small k, magnitude ≈ −0.05 to −0.15, fading as k grows.

➜ **One sentence:** you have just measured, in advance, the failure mode of the model you will fit tomorrow. Name it.

_Your sentence:_

## §2.5 Aggregational Gaussianity — guide §2.5 · *(cuttable — ROADMAP cut #1)*

- Q. The CLT grinds aggregated returns toward the Gaussian. Which two features you documented today slow it down?

**Task:** excess kurtosis at 1-day, 5-day, 21-day horizons. Summing log returns over blocks is licensed by the additivity you tested this morning.

<details><summary>Hint — non-overlapping blocks</summary>

Group positions into blocks (`np.arange(len(r)) // 5`) and sum within blocks — check you are not accidentally using overlapping windows, which would smooth the answer.

</details>

In [ ]:
# TODO(you): excess kurtosis at 1d / 5d / 21d

### ✓ Checkpoint 6

- Kurtosis falls with horizon — toward ≈ 1–5 at 21 days, still above the Gaussian's 0.

➜ **One sentence** tying this to the CLT and to exactly which of its assumptions your data weaken.

_Your sentence:_

## §3 Rolling realized volatility — guide §3

- Q1. Your estimator is a boxcar filter on the r² series. What does the window length trade off?
- Q2. Where does √252 come from — and which stylized fact from §2.2 earned you the right to use it?
- Q3. Why is the mean dropped? (Look back at Checkpoint 1.)

**Task** — implement `rolling_vol` in [src/garch_lab/volatility.py](../src/garch_lab/volatility.py).

**DONE when:** `pytest tests/test_volatility.py` is green. Then plot the 21-day annualized vol over the full sample, overlay the 5-day and 63-day variants, and export `figures/fig3_rolling_vol.png` (a draft — tomorrow's version adds the GARCH overlay and takes its README slot).

Vocabulary for your write-up: call yours **"21-day rolling volatility"**, not "realized volatility" — in the literature RV means intraday sums (guide §3.2). Naming it precisely signals you know the difference.

<details><summary>Hint 1 — concept</summary>

Root of the rolling *mean* of squared returns — not a rolling standard deviation. One of the tests exists purely to catch that difference.

</details>

<details><summary>Hint 2 — API</summary>

`Series.rolling(window).mean()` on a squared series, `np.sqrt`, and the constant already defined in the module.

</details>

In [ ]:
# TODO(you): green tests; 21d annualized vol plot; overlay 5d and 63d;
#            export figures/fig3_rolling_vol.png

### ✓ Checkpoint 7

- Long-run median ≈ 12–18% annualized; calm 2017 ≈ 6–10%; 2008 and 2020 peaks ≈ 70–90%.
- Debug tells: everything off by ×15.9 → you forgot the square root of 252; off by ×100 → percent vs raw units.

➜ **One sentence:** which window would you hand a risk desk, and what did you trade away to get it?

_Your sentence:_

## Wrap — what you proved today

Collect your seven checkpoint sentences here; they are the draft of the README section "What the data demand of a model" (ROADMAP box S4):

1. …
2. …

**Teaser for tomorrow:** your boxcar weights day 21 fully and day 22 at zero, and the window length was chosen by convention, not by the data. GARCH replaces it with an exponentially decaying filter whose decay rate is *estimated by maximum likelihood* — plus mean reversion. Same signal-processing object, better impulse response (guide §3.2 → §4).